# Decision Model 实验：结果复算与 Demo

本 notebook 从导出的结果 ZIP 复算统计、显示图表和回放真实记录。默认只用 CPU，不启动模型服务。
图表来自结果包中的逐样本预测；结果不完整时停止分析并列出缺失项，不补造数据。

需要 `decision-study-bundle.zip`（代码）和 pilot／experiment／benchmark 结果 ZIP。

## 1. Parameters

用新的 `OUTPUT_DIR` 导入一个结果包，避免混入其他实验。`RESULT_ZIP` 留空时，Colab 会提示上传；
VM 上请填写实际文件路径。已经导入过的结果或原始实验目录可设 `USE_EXISTING_OUTPUT=True`，
不再覆盖导入；实时 demo 使用这一模式，仅在最后一节显式开启。

In [ ]:
PLATFORM = "code-server"                   # "code-server", "colab" or "vm"
PACKAGE_DIR = ""
PACKAGE_ZIP = ""
RESULT_ZIP = ""
OUTPUT_DIR = ""                            # blank: ./decision-study-analysis
USE_EXISTING_OUTPUT = False                 # True for reruns or the original experiment directory; no ZIP import
INSTALL_PACKAGE = True
INSTALL_RUNTIME_TOOLS = False              # CPU result analysis does not need uv
RECOMPUTE = True                            # formal analysis uses 2,000 bootstrap replicates
DEMO_MODEL = "kev"                         # "kev" or "startlux"
SAMPLE_ID = None                            # None: choose a saved case deterministically
RUN_LIVE = False                            # requires compatible GPU + frozen model/gate configuration
LIVE_TASK = "clinc"                        # "clinc" or "squad"
LIVE_TEXT = "I need to check my account balance."
LIVE_PASSAGE = "The museum opened in 1984."
LIVE_QUESTION = "When did the museum open?"

## 2. Load the same experiment code

In [ ]:
from pathlib import Path
import importlib
import os
import stat
import subprocess
import sys
import tempfile
import zipfile

def locate_source(explicit=""):
    candidates = ([Path(explicit).expanduser()] if explicit else []) + [
        Path.cwd(), Path.cwd().parent,
        Path.cwd() / "decision-study", Path.cwd().parent / "decision-study",
    ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "decision_study").is_dir():
            return candidate.resolve()
    return None

def extract_source_zip(archive):
    destination = Path(tempfile.mkdtemp(prefix="decision-study-source-"))
    with zipfile.ZipFile(archive) as bundle:
        if sum(item.file_size for item in bundle.infolist()) > 512 * 1024**2:
            raise ValueError("Source archive exceeds 512 MiB. Use the code bundle, not model weights.")
        for item in bundle.infolist():
            name = item.filename.replace("\\", "/")
            relative = Path(name)
            if (relative.is_absolute() or ".." in relative.parts or
                    (relative.parts and ":" in relative.parts[0]) or
                    stat.S_ISLNK(item.external_attr >> 16)):
                raise ValueError(f"Unsafe archive entry: {item.filename}")
            target = (destination / relative).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError("Archive path escapes extraction directory.")
        bundle.extractall(destination)
    candidates = [destination] + [p for p in destination.iterdir() if p.is_dir()]
    matches = [p for p in candidates if (p / "pyproject.toml").is_file() and (p / "decision_study").is_dir()]
    if len(matches) != 1:
        raise ValueError("Expected one decision-study project in the source ZIP.")
    return matches[0].resolve()

PROJECT_DIR = locate_source(PACKAGE_DIR)
if PROJECT_DIR is None:
    archive = Path(PACKAGE_ZIP).expanduser() if PACKAGE_ZIP else Path.cwd() / "decision-study-bundle.zip"
    if not archive.is_file():
        if PLATFORM != "colab":
            raise FileNotFoundError("Set PACKAGE_DIR to the extracted project or PACKAGE_ZIP to decision-study-bundle.zip.")
        from google.colab import files
        print("Upload decision-study-bundle.zip (the code package).")
        uploaded = files.upload()
        candidates = [Path(name) for name in uploaded if name.endswith(".zip") and "decision-study" in name]
        if len(candidates) != 1:
            raise ValueError("Upload exactly one decision-study code ZIP.")
        archive = candidates[0]
    PROJECT_DIR = extract_source_zip(archive)
if INSTALL_PACKAGE:
    install_command = [sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR)]
    if globals().get("INSTALL_RUNTIME_TOOLS", False):
        install_command.append("uv")
    subprocess.run(install_command, check=True)
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")
sys.path.insert(0, str(PROJECT_DIR))
importlib.invalidate_caches()
print("Experiment package:", PROJECT_DIR)

## 3. Import the verified result bundle

Import checks the bundle manifest and file checksums. Model weights and credentials are not part of the result bundle.
Status remains explicit if only the pilot or a subset of formal jobs is present.

In [ ]:
import json
from decision_study.artifacts import import_bundle, bundle_readiness
result_dir = (Path(OUTPUT_DIR).expanduser() if OUTPUT_DIR else Path.cwd() / "decision-study-analysis").resolve()
if USE_EXISTING_OUTPUT:
    from decision_study.data import verify_data
    verified = verify_data(result_dir)
    imported = {"mode": "existing output; no import or overwrite", "data_fingerprint": verified["fingerprint"]}
else:
    if RESULT_ZIP:
        archive = Path(RESULT_ZIP).expanduser()
    elif PLATFORM == "colab":
        from google.colab import files
        print("Upload one pilot_bundle.zip, experiment_bundle.zip or benchmark_bundle.zip.")
        uploaded = files.upload()
        candidates = [Path(name) for name in uploaded if name.endswith("_bundle.zip")]
        if len(candidates) != 1:
            raise ValueError("Upload exactly one result bundle.")
        archive = candidates[0]
    else:
        raise ValueError("Set RESULT_ZIP, or use USE_EXISTING_OUTPUT=True with a prepared results directory.")
    imported = import_bundle(archive, result_dir)
readiness = bundle_readiness(result_dir)
print(json.dumps({"import": imported, "readiness": readiness}, ensure_ascii=False, indent=2))

## 4. Recompute statistics on CPU

Calibration and validation gates remain frozen. Test uncertainty uses paired resampling (article clusters for SQuAD).
The evaluator keeps failures in accuracy denominators and reports probability metrics only for valid distributions.
This cell does not make model calls.

In [ ]:
if readiness["experiment_ready"] and RECOMPUTE:
    subprocess.run([sys.executable, "-m", "decision_study", "evaluate", "--output-dir", str(result_dir)],
                   cwd=PROJECT_DIR, check=True)
    print("Recomputed from saved predictions. No GPU inference was run.")
elif not readiness["experiment_ready"]:
    print("Analysis skipped: the formal experiment is incomplete.")
    print(json.dumps(readiness.get("missing", []), ensure_ascii=False, indent=2))
else:
    print("Recomputation disabled; figures below, if any, are the saved bundle artifacts.")

## 5. Inspect result figures and exact-value tables

Read risk and coverage together. Calibration improvements need not improve ranking, especially in the binary task.
Cached predictions support accuracy analysis; only live benchmark traces support latency claims.

In [ ]:
from IPython.display import display, Image, Markdown
import csv
if readiness["experiment_ready"]:
    presentation_dirs = [result_dir / "analysis"]
    if readiness["benchmark_ready"]:
        presentation_dirs.append(result_dir / "benchmark")
    figures = sorted(p for folder in presentation_dirs for p in folder.rglob("*.png"))
    for figure in figures:
        display(Markdown(f"**{figure.stem.replace('_', ' ')}**"))
        display(Image(filename=str(figure), width=950))
    if not figures:
        print("No analysis figures found; inspect the evaluate output above.")
    for table in sorted(p for folder in presentation_dirs for p in folder.rglob("*.csv")):
        with table.open(encoding="utf-8", newline="") as handle:
            reader = csv.DictReader(handle)
            preview = [row for _, row in zip(range(5), reader)]
        print("Table:", table.relative_to(result_dir), "(first 5 rows)")
        display(preview)
else:
    print("No formal result figures displayed for an incomplete experiment.")

## 6. Recorded demo — explicitly a replay

The example below comes from saved predictions and the frozen gate. It is not a fresh model call.
Change `SAMPLE_ID` to inspect a particular input. Replays can show failures and harmful fallback as well as successful decisions.

In [ ]:
from decision_study.demo import replay
if readiness["experiment_ready"]:
    replay_result = replay(result_dir, DEMO_MODEL, sample_id=SAMPLE_ID)
    print("RECORDED REPLAY — no live inference")
    display(replay_result)
else:
    print("Replay skipped: complete the formal prediction jobs first.")

## 7. Optional live demo

Enable `RUN_LIVE=True` only on the **original matched experiment GPU/runtime**, with its model environments and weights still available,
after completing calibration and smoke. Set `OUTPUT_DIR` to that original experiment directory and `USE_EXISTING_OUTPUT=True`.
A result ZIP imported into a fresh CPU machine does not install the GPU environments or weights and is sufficient only for analysis/replay.
This starts real decision and Qwen services and consumes GPU time. It uses the same adapters and frozen gate as the experiments.
Enter text for CLINC, or a complete passage and question for SQuAD. Stop the VM when finished.

In [ ]:
if RUN_LIVE:
    if not USE_EXISTING_OUTPUT:
        raise RuntimeError("For live inference, use the original experiment OUTPUT_DIR with USE_EXISTING_OUTPUT=True.")
    if not readiness["experiment_ready"]:
        raise RuntimeError("Live demo requires the completed experiment and frozen gates.")
    if PLATFORM == "colab":
        os.environ["DECISION_STUDY_RUNTIME_DIR"] = "/content/decision-study-runtime"
    from decision_study.demo import live
    display(live(PROJECT_DIR, result_dir, DEMO_MODEL, LIVE_TASK,
                 text=LIVE_TEXT, passage=LIVE_PASSAGE, question=LIVE_QUESTION))
else:
    print("Live demo disabled. This notebook has only inspected/recomputed saved results.")

## Send back / write up

Keep the original result ZIP plus the generated `analysis` directory. For interpretation, return the full per-example bundle,
not only screenshots or summary accuracy. The next discussion can use the plots, paired differences, selected errors and live
cost traces to decide the report's main findings. No positive result is assumed.